# Euclid cutouts with Cutana
Matches sources from a CSV catalogue to Euclid tiles (DR1 R2 → DR1 R1 → Q1 R1, in priority order), builds a Cutana-compatible source catalogue, and launches the Cutana UI.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_CSV       = "/home/mgrespan/my_workspace/euclid_cutouts/my_search_sample.csv"
ID_COL          = "object_id"
CUTOUT_PIXELS   = 128                  # cutout size in pixels
OUTPUT_DIR      = "/home/mgrespan/my_workspace/euclid_cutouts/cutana_output"
TILE_CENTRES    = "/home/mgrespan/my_workspace/euclid_cutouts/tile_centres.csv"

BANDS           = ["NIR_Y", "NIR_J", "VIS"]   # Cutana band order
BAND_TO_INST    = {"NIR_Y": "NISP", "NIR_J": "NISP", "VIS": "VIS"}

# RA/Dec column auto-detection (case-insensitive).
# Accepted names: ra / right_ascension / target_ra,  dec / declination / target_dec
_RA_ALIASES  = ["ra", "right_ascension", "target_ra"]
_DEC_ALIASES = ["dec", "declination", "target_dec"]
# ── END CONFIG ────────────────────────────────────────────────────────────────

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial import KDTree
from tqdm import tqdm
tqdm.pandas()

sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "scripts"))
from fits_path_utils import find_fits_paths

import cutana_ui

## Step 0: Load source catalogue

In [ ]:
df = pd.read_csv(INPUT_CSV)

# auto-detect RA / Dec columns
cols_lower = {c.lower().strip(): c for c in df.columns}
RA_COL  = next((cols_lower[a] for a in _RA_ALIASES if a in cols_lower), None)
DEC_COL = next((cols_lower[a] for a in _DEC_ALIASES if a in cols_lower), None)
assert RA_COL and DEC_COL, f"Could not find RA/Dec columns in {list(df.columns)}"

print(f"Loaded {len(df)} sources  (RA={RA_COL}, Dec={DEC_COL})")
df.head()

## Step 1: Match sources to tiles
Uses a pre-built tile centre table and a KDTree for fast nearest-tile lookup.
Priority: DR1 R2 → DR1 R1 → Q1 R1.

In [4]:
tiles = pd.read_csv(TILE_CENTRES)

# convert to 3D unit vectors for correct spherical KDTree matching
def to_xyz(ra_deg, dec_deg):
    ra  = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)
    return np.column_stack([np.cos(dec)*np.cos(ra),
                            np.cos(dec)*np.sin(ra),
                            np.sin(dec)])

tile_xyz = to_xyz(tiles["ra"].values, tiles["dec"].values)
tree = KDTree(tile_xyz)

src_xyz = to_xyz(df[RA_COL].values, df[DEC_COL].values)
dist, idx = tree.query(src_xyz)

# dist is chord length; convert to angular separation in degrees
ang_sep = np.rad2deg(2 * np.arcsin(dist / 2))

matched = tiles.iloc[idx].reset_index(drop=True)
df = df.copy().reset_index(drop=True)
df["tile_index"]   = matched["tile_index"].values
df["release_dir"]  = matched["release_dir"].values
df["half_size_deg"]= matched["half_size_deg"].values

# keep only sources within the tile footprint (with small buffer)
in_tile = ang_sep < (matched["half_size_deg"].values * np.sqrt(2) + 0.05)
before = len(df)
df = df[in_tile].copy()
print(f"{len(df)}/{before} sources matched to tiles")
df["release"].value_counts() if "release" in df else df["release_dir"].apply(lambda x: x.split("/")[-2]+"/"+x.split("/")[-1]).value_counts()

6993321/6993321 sources matched to tiles


release_dir
DR1/R1             4948747
DR1/R2             2033819
euclid_q1/Q1_R1      10755
Name: count, dtype: int64

## Step 2: Build the Cutana source catalogue
Cutana expects columns: `SourceID`, `RA`, `Dec`, `diameter_pixel`, `fits_file_paths`.
Band order is **NIR-Y, NIR-J, VIS** — must be consistent across all rows.

In [ ]:
def identify_observations(row):
    paths = find_fits_paths(
        int(row["tile_index"]), BANDS, row["release_dir"],
        BAND_TO_INST, ra=row[RA_COL], dec=row[DEC_COL],
    )
    return str(paths) if paths else None

df_cutana = (
    df
    .rename(columns={ID_COL: "SourceID", RA_COL: "RA", DEC_COL: "Dec"})
    [["SourceID", "RA", "Dec", "tile_index", "release_dir"]]
    .assign(
        SourceID        = lambda d: d["SourceID"].astype(str).str.replace("-", "NEG", regex=False),
        diameter_pixel  = CUTOUT_PIXELS,
        fits_file_paths = lambda d: d.progress_apply(identify_observations, axis=1),
    )
    .dropna(subset=["fits_file_paths"])
    .drop(columns=["tile_index", "release_dir"])
)

print(f"{len(df_cutana)} sources ready for Cutana")
df_cutana.head()

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
catalogue_path = os.path.join(OUTPUT_DIR, "cutana_catalogue.csv")
df_cutana.to_csv(catalogue_path, index=False)
print(f"Saved catalogue to {catalogue_path}")

## Step 3: Run Cutana
Select the catalogue saved above and set the output folder to `OUTPUT_DIR`.

In [ ]:
cutana_ui.start(ui_scale=0.75)

## Step 4: Preview cutouts from zarr output

In [ ]:
zarr_files = glob.glob(os.path.join(OUTPUT_DIR, "batch_cutout*"))

file = zarr.open(f"{zarr_files[0]}/images.zarr/", mode="r")
n_images = file["images"].shape[0]

ncols = min(10, n_images)
nrows = int(np.ceil(n_images / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.6, nrows * 1.6))
for i, ax in enumerate(np.array(axes).flatten()):
    if i < n_images:
        ax.imshow(file["images"][i])
    ax.axis("off")
plt.tight_layout()
plt.show()

## Step 5: Export cutouts to JPEG

In [ ]:
jpeg_folder = os.path.join(OUTPUT_DIR, "jpegs")
os.makedirs(jpeg_folder, exist_ok=True)

for zarr_file in zarr_files:
    file     = zarr.open(f"{zarr_file}/images.zarr/", mode="r")
    metadata = pd.read_parquet(f"{zarr_file}/images_metadata.parquet")
    for i in tqdm(range(file["images"].shape[0])):
        source_id = metadata.source_id.iloc[i]
        Image.fromarray(file["images"][i]).save(f"{jpeg_folder}/{source_id}.jpg")

print(f"Saved {len(glob.glob(jpeg_folder + '/*.jpg'))} JPEGs to {jpeg_folder}")